# Manual Price Push

Push manual price overrides for specific SKUs using market data or fixed prices.

## How to use
1. Run cells 1-3 (setup + data loading)
2. Edit the `PUSH_LIST` in cell 4 with your SKUs and desired actions
3. Run cell 4 to compute new prices
4. Review the output table
5. Set `MODE = 'live'` and run cell 5 to push

## Available Actions
| Action | Description |
|--------|-------------|
| `market_min` | Lowest market price (P0 / minimum) |
| `market_25` | 25th percentile market price |
| `market_50` | Median market price (P50) |
| `market_75` | 75th percentile market price |
| `market_max` | Highest market price (maximum) |
| `market_avg` | Average of min and max market prices |
| `target_margin` | Price from brand-category target margin |
| `<number>` | Fixed price (e.g. `115`, `23.5`) |

In [1]:
# =============================================================================
# SETUP
# =============================================================================
import sys, os
import pandas as pd
import numpy as np
import pytz
from datetime import datetime

sys.path.insert(0, os.path.abspath('.'))
import setup_environment_2
setup_environment_2.initialize_env()

from db import query_snowflake
from constants import WAREHOUSE_MAPPING, COHORT_IDS

os.chdir(os.path.join(os.path.dirname(os.path.abspath('__file__')) or os.getcwd(), 'modules'))
%run push_prices_handler.ipynb
os.chdir('..')

CAIRO_TZ = pytz.timezone('Africa/Cairo')
TODAY = datetime.now(CAIRO_TZ).strftime('%Y-%m-%d')
print(f'Ready. Today: {TODAY}')

/home/ec2-user/.Renviron
/home/ec2-user/service_account_key.json


/home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/snowflake/connector/options.py:104: UserWarning: You have an incompatible version of 'pyarrow' installed (20.0.0), please install a version that adheres to: 'pyarrow<19.0.0; extra == "pandas"'
  warn_incompatible_dep(


Push Prices Handler loaded at 2026-04-02 15:03:16 Cairo time
✓ API credentials loaded successfully
✓ Google Sheets client initialized
Ready. Today: 2026-04-02


In [5]:
# =============================================================================
# LOAD MARKET DATA + WAC + CURRENT PRICES + PACKING UNITS
# =============================================================================
os.chdir('modules')
%run market_data_module.ipynb
os.chdir('..')

print('Loading market data...')
market_data = get_market_data()
print(f'  Market data: {len(market_data)} rows')

print('Loading WAC...')
WAC_QUERY = """
SELECT product_id, wac_p
FROM finance.all_cogs
WHERE wac_p > 0
"""
df_wac = query_snowflake(WAC_QUERY)
print(f'  WAC: {len(df_wac)} rows')

print('Loading current prices...')
PRICES_QUERY = f"""
WITH ranked AS (
    SELECT
        cpu.cohort_id,
        pu.product_id,
        pu.packing_unit_id,
        pu.basic_unit_count,
        cpu.price AS current_price,
        ROW_NUMBER() OVER (PARTITION BY cpu.cohort_id, pu.product_id, pu.packing_unit_id
                           ORDER BY cpu.created_at DESC) AS rn
    FROM cohort_product_packing_units cpu
    JOIN packing_unit_products pu ON pu.id = cpu.product_packing_unit_id
    WHERE cpu.cohort_id IN ({','.join(str(c) for c in COHORT_IDS)})
      AND cpu.is_customized = TRUE
)
SELECT cohort_id, product_id, packing_unit_id, basic_unit_count, current_price
FROM ranked WHERE rn = 1
"""
df_prices = query_snowflake(PRICES_QUERY)
print(f'  Prices: {len(df_prices)} rows')

print('Loading packing units...')
PU_QUERY = """
SELECT DISTINCT
    pu.product_id,
    pu.packing_unit_id,
    pu.basic_unit_count
FROM packing_unit_products pu
WHERE pu.deleted_at is null
"""
df_pus = query_snowflake(PU_QUERY)
print(f'  Packing units: {len(df_pus)} rows')

print('Loading product info...')
PRODUCT_QUERY = """
SELECT p.id AS product_id,
       CONCAT(p.name_ar, ' ', p.size, ' ', pu.name_ar) AS sku,
       b.name_ar AS brand,
       c.name_ar AS cat
FROM products p
JOIN brands b ON b.id = p.brand_id
JOIN categories c ON c.id = p.category_id
JOIN product_units pu ON pu.id = p.unit_id
"""
df_products = query_snowflake(PRODUCT_QUERY)
print(f'  Products: {len(df_products)} rows')

print('Loading target margins...')
MARGIN_QUERY = """
WITH latest AS (
    SELECT brand, cat, margin as target_margin,
           ROW_NUMBER() OVER (PARTITION BY brand, cat ORDER BY date DESC) AS rn
    FROM performance.commercial_targets
    WHERE target_margin IS NOT NULL
)
SELECT brand, cat, target_margin FROM latest WHERE rn = 1
"""
df_targets = query_snowflake(MARGIN_QUERY)
print(f'  Target margins: {len(df_targets)} rows')

print('\nAll data loaded.')

/home/ec2-user/.Renviron
/home/ec2-user/service_account_key.json
Market Data Module loaded at 2026-04-02 15:29:13 Cairo time
Snowflake timezone: America/Los_Angeles
All queries defined ✓
Helper functions defined ✓
get_market_data() function defined ✓
get_margin_tiers() function defined ✓
get_brand_market_percentiles() function defined ✓
fill_brand_market_fallback() function defined ✓

MARKET DATA MODULE READY

Available functions (NO INPUT REQUIRED):
  - get_market_data()              : Fetch and process all market prices
  - get_margin_tiers()             : Fetch and calculate margin tiers
  - get_brand_market_percentiles() : Fetch brand-level market margin percentiles
  - fill_brand_market_fallback()   : Fill NaN market cols with brand percentiles

Usage:
  %run market_data_module.ipynb
  df_market = get_market_data()
  df_tiers = get_margin_tiers()
  df_brand_percs = get_brand_market_percentiles()
  df = fill_brand_market_fallback(df, df_brand_percs)
Loading market data...

FETCHING

/tmp/ipykernel_1120/3473350795.py:139: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({


    Records after group processing: 25885

Step 5: Adding WAC and margin data...
    Records with WAC: 25525

Step 6: Filtering by price coverage...
    Records after price coverage filter: 15515

Step 7: Calculating price percentiles...
    Records after price analysis: 14904

Step 8: Converting prices to margins...

MARKET DATA COMPLETE
Total records: 14904
  - With marketplace prices: 12572
  - With scrapped prices: 8576
  - With Ben Soliman prices: 12029
  Market data: 14904 rows
Loading WAC...
  WAC: 948836 rows
Loading current prices...
  Prices: 111980 rows
Loading packing units...
  Packing units: 36037 rows
Loading product info...
  Products: 25595 rows
Loading target margins...
  Target margins: 738 rows

All data loaded.


In [6]:
# =============================================================================
# BUILD LOOKUP TABLE (product_id → market prices + WAC + target margin)
# =============================================================================

price_cols = ['minimum', 'percentile_25', 'percentile_50', 'percentile_75', 'maximum']
market_subset = market_data[['product_id', 'cohort_id'] + [c for c in price_cols if c in market_data.columns]].copy()

lookup = df_products.merge(df_wac, on='product_id', how='left')
lookup = lookup.merge(
    market_subset.drop_duplicates(subset='product_id', keep='first'),
    on='product_id', how='left'
)
lookup = lookup.merge(df_targets, on=['brand', 'cat'], how='left')
lookup['target_margin'] = lookup['target_margin'].fillna(0.10)

if 'minimum' in lookup.columns and 'maximum' in lookup.columns:
    lookup['market_avg'] = (lookup['minimum'] + lookup['maximum']) / 2

print(f'Lookup table: {len(lookup)} products')
print(f'  With market data: {lookup["minimum"].notna().sum()}')
print(f'  With WAC: {lookup["wac_p"].notna().sum()}')
print(f'  With target margin: {lookup["target_margin"].notna().sum()}')

Lookup table: 966039 products
  With market data: 437116
  With WAC: 948836
  With target margin: 966039


In [11]:
# =============================================================================
# DEFINE YOUR SKU ACTIONS HERE
# =============================================================================
# Format: (product_id, action)
#   action can be: 'market_min', 'market_25', 'market_50', 'market_75',
#                  'market_max', 'market_avg', 'target_margin', or a number

PUSH_LIST = [
    # (product_id, action)
    (6935, 'market_50')
    # (5678, 'market_50'),
    # (9999, 115),            # fixed price = 115 EGP per basic unit
    # (4444, 'target_margin'),
]


# =============================================================================
# COMPUTE NEW PRICES
# =============================================================================
ACTION_TO_COLUMN = {
    'market_min': 'minimum',
    'market_25': 'percentile_25',
    'market_50': 'percentile_50',
    'market_75': 'percentile_75',
    'market_max': 'maximum',
    'market_avg': 'market_avg',
}

results = []
for product_id, action in PUSH_LIST:
    row = lookup[lookup['product_id'] == product_id]
    if row.empty:
        results.append({'product_id': product_id, 'action': str(action), 'status': 'NOT FOUND'})
        continue

    row = row.iloc[0]
    wac = row.get('wac_p', None)
    sku = row.get('sku', '')
    brand = row.get('brand', '')

    if isinstance(action, (int, float)):
        base_price = float(action)
        action_label = f'fixed_{action}'
    elif action == 'target_margin':
        tm = row.get('target_margin', 0.10)
        if pd.isna(wac) or wac <= 0:
            results.append({'product_id': product_id, 'sku': sku, 'action': action, 'status': 'NO WAC'})
            continue
        base_price = wac / (1 - tm)
        action_label = f'target_margin ({tm:.1%})'
    elif action in ACTION_TO_COLUMN:
        col = ACTION_TO_COLUMN[action]
        val = row.get(col, None)
        if pd.isna(val):
            results.append({'product_id': product_id, 'sku': sku, 'action': action, 'status': 'NO MARKET DATA'})
            continue
        base_price = val
        action_label = action
    else:
        results.append({'product_id': product_id, 'sku': sku, 'action': str(action), 'status': 'INVALID ACTION'})
        continue

    base_price_rounded = np.round(base_price * 4) / 4

    margin = (base_price_rounded - wac) / base_price_rounded if (wac and wac > 0 and base_price_rounded > 0) else None

    results.append({
        'product_id': product_id,
        'sku': sku,
        'brand': brand,
        'action': action_label,
        'wac': wac,
        'base_price': base_price,
        'new_price': base_price_rounded,
        'margin': margin,
        'status': 'OK',
    })

df_results = pd.DataFrame(results)

ok_count = (df_results['status'] == 'OK').sum()
err_count = len(df_results) - ok_count
print(f'Results: {ok_count} OK, {err_count} errors/warnings')
if err_count > 0:
    print('\nErrors:')
    print(df_results[df_results['status'] != 'OK'][['product_id', 'sku', 'action', 'status']].to_string(index=False))
print()
df_results[df_results['status'] == 'OK']

Results: 1 OK, 0 errors/warnings



,product_id,sku,brand,action,wac,base_price,new_price,margin,status
0,6935,كوكاكولا اكشن - 300 مل,كوكا كولا,market_50,94.89,117.6,117.5,0.192426,OK


In [15]:
# =============================================================================
# PUSH PRICES
# =============================================================================
MODE = 'live'  # Change to 'live' to actually push

df_ok = df_results[df_results['status'] == 'OK'].copy()
if len(df_ok) == 0:
    print('No valid prices to push.')
else:
    wh_df = pd.DataFrame(WAREHOUSE_MAPPING, columns=['region', 'warehouse', 'warehouse_id', 'cohort_id'])

    push_rows = []
    for _, r in df_ok.iterrows():
        for _, wh in wh_df.iterrows():
            push_rows.append({
                'product_id': int(r['product_id']),
                'sku': r['sku'],
                'new_price': r['new_price'],
                'warehouse_id': int(wh['warehouse_id']),
                'cohort_id': int(wh['cohort_id']),
                'stocks': 1,
                'current_price': r['new_price']-1,
            })

    df_push = pd.DataFrame(push_rows)

    print(f'Pushing {len(df_ok)} products × {len(wh_df)} warehouses = {len(df_push)} rows')
    print(f'Mode: {MODE}')
    print()

    result = push_prices(
        df_prices=df_push,
        pus=df_pus,
        source_module='manual_push',
        mode=MODE,
    )
    print(f'\nDone. Result: {result}')

Pushing 1 products × 12 warehouses = 12 rows
Mode: live


🚀 MODE: LIVE
   Files will be prepared AND uploaded to API
Loading disable_pu_visibility from Google Sheets...
  ✓ Loaded 88 products to disable min PU visibility

PUSH PRICES - Source: manual_push
Total received: 12
Price changes to push: 12
Skipped (no change): 0

Price changes summary:
  Increases: 12
  Decreases: 0

🔗 Mirrored prices to 6 main/general cohorts (+6 rows)
   Cohort 695 ← 700: 1 rows
   Cohort 61 ← 700: 1 rows
   Cohort 699 ← 702: 1 rows
   Cohort 697 ← 703: 1 rows
   Cohort 698 ← 704: 1 rows
   Cohort 696 ← 1123: 1 rows

📋 Prepared 15 packing unit prices (incl. main cohorts)

Processing cohort: 61
  Saved: uploads/manual_push_61.xlsx (1 rows)
  Split into 1 chunks (size: 2000)


  Saving chunks: 100%|██████████| 1/1 [00:00<00:00, 220.59it/s]

  Uploading...


    ✓ Chunk 1 uploaded successfully

Processing cohort: 695
  Saved: uploads/manual_push_695.xlsx (1 rows)
  Split into 1 chunks (size: 4000)


  Saving chunks: 100%|██████████| 1/1 [00:00<00:00, 233.71it/s]

...

  Uploading...


    ✓ Chunk 1 uploaded successfully

Processing cohort: 1126
  Saved: uploads/manual_push_1126.xlsx (1 rows)
  Split into 1 chunks (size: 4000)


  Saving chunks: 100%|██████████| 1/1 [00:00<00:00, 240.17it/s]

  Uploading...


    ✓ Chunk 1 uploaded successfully

🚀 UPLOAD COMPLETE
Mode: live
Total prepared: 15
Total failed: 0
  Cleanup: removed 46 temporary .xlsx files from 2 folder(s)

Done. Result: {'total_received': 12, 'price_changes': 12, 'pushed': 15, 'failed': 0, 'skipped': 0, 'source_module': 'manual_push', 'timestamp': '2026-04-02 15:36:05', 'mode': 'live', 'skipped_cohorts': []}
